# Assignment APIs tutorial

In this notebook we are using the python package "millionaire-client" to interact with the deployed application for the NLP assignment 2026.

Required files:
- Directory called "millionaire_client"
- 1 colab notebook

Both files must be saved in a directory in your Google Drive, for example:
```
gDrive_home/
├── Colab Notebooks/
│   └── NLP_assignment/
│       ├── PoliMillionaire.ipynb <-- Your notebook
│       └── millionaire_client/ <-- Directory provided
```

### Sign up procedure
Before showing you how the api work, you need to signup from a web browser.
- Paste this link into your browser [http://131.175.15.22:51111/](http://131.175.15.22:51111/) this is where the demo is deployed
- You will see a standard login/sign up screen, please click on sign up
- In the "email" field please enter your politecnico email, you are allwed to create only 1 account using the same email you registered to the NLP course
- Choose whever username/password you prefer (be creative ;))

### Game interaction

Once you signed up, you can start interacting already from the api.

First of all, let's connect your drive to this Colab Notebook

In [1]:
from google.colab import drive
import os
drive.mount('/content/gdrive/')

Mounted at /content/gdrive/


Then we need to add our python package "millionaire_client" to the system path, so python can see it.

In [2]:
import sys
import os

# Define the path to the directory containing your package
package_parent_dir = '/content/gdrive/MyDrive/NLP_assignment/'

# Append to sys.path if it is not already present
if package_parent_dir not in sys.path:
    sys.path.append(package_parent_dir)

# Verify the path was added
print(sys.path)

['/content', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.12/dist-packages/IPython/extensions', '/root/.ipython', '/content/gdrive/MyDrive/NLP_assignment/']


Let's import the client classes

In [3]:
from millionaire_client import MillionaireClient, AuthenticationError

You can save your password in a Colab secret (the "key" icon on the tab on the left) and import it into your notebook.

In [ ]:
#from google.colab import userdata
#pwd = userdata.get('poli-millionaire')

Now keep the API_URL as stated, but please change the username and password to be the ones you used during sign up session.

In [ ]:
API_URL = "http://131.175.15.22:51111/"
username = ""
password = ""

Now we can instantiate a MillionaireClient object and call the login method, which takes as parameters username and password.

In [5]:
client = MillionaireClient(API_URL)
try:
    user = client.login(username, password)
    print(f"\nWelcome, {user.username}! (Role: {user.role})")
except AuthenticationError as e:
    print(f"Login failed: {e}")


Welcome, gary! (Role: student)


After login, the web page is showing you different types of competitions, for each of them you can choose to play a game or to see the leaderboard. For now let's list all of the.

In [6]:
# List available competitions
print("\n=== Available Competitions ===")
competitions = client.competitions.list_all()
for comp in competitions:
    print(f"  {comp.id}: {comp.name} ({comp.max_levels} questions)")


=== Available Competitions ===
  0: Entertainment (15 questions)
  1: Ancient History and Politics (15 questions)
  2: Science and Nature (15 questions)
  3: Maths (15 questions)


In [7]:
# Choose a competition ID
comp_id = 1

After choosing a competition, we can start a game! We can choose to start a game by calling `game = client.game.start(competition_id=comp_id)`. The object game is the one that is handling the game itself, we can call:
- game.current_question.text : to know the current question in text format
- game.current_level: to check the current level of difficulty of the question
- game.current_question.options: to check the possible choices we have to answer the question
- game.answer: to send to the server the answer we choose (the integer corresponding to our choice) and get the response (either correct or incorrect)

WATCH OUT! Each question has a timer, you have maximum 30 seconds to answer the question. As of now, if you exceed the maximum allowed time, there is not a "push notification". You still have to submit your answer anyway and, even though the answer was correct, you will get a TimedOut response!

In [23]:
def play_game(game):
  # Play the game
  while game.in_progress:
      question = game.current_question
      if not question:
          print("No question available. Game may have ended.")
          break

      print(f"\n--- Level {game.current_level} ---")
      print(f"Q: {question.text}")
      print()

      for opt in question.options:
          print(f"  [{opt.id}] {opt.text}")

      # Get time remaining
      time_left = game.time_remaining
      if time_left:
          print(f"\nTime remaining: {time_left:.1f}s")

      # Get answer
      try:
          answer_input = input("\nYour answer (option ID): ").strip()
          answer_id = int(answer_input)
      except ValueError:
          print("Invalid input. Please enter a number.")
          continue

      # Submit answer
      result = game.answer(answer_id)

      if result.correct:
          print(" CORRECT!")
          if result.game_over:
              print(f"\n CONGRATULATIONS! You completed the game!")
              print(f" Final earnings: ${result.earned_amount:,.2f}")
          else:
              print(f" Earned so far: ${result.earned_amount:,.2f}")
      elif result.timed_out:
        print("TIMED OUT!")
        print(f"\n Game Over!")
        print(f" Final earnings: ${result.earned_amount:,.2f}")
      elif not result.correct:
          print(" WRONG ANSWER!")
          print(f"\n Game Over!")
          print(f" Final earnings: ${result.earned_amount:,.2f}")

  print("\n=== Game Summary ===")
  print(f"Reached Level: {game.current_level}")
  print(f"Total Earnings: ${game.earned_amount:,.2f}")

In [84]:
# Start the game
print("\n=== Starting Game ===")
game = client.game.start(competition_id=comp_id)
print(f"Session ID: {game.session_id}")
print(f"Total number of questions: {game.state.competition.max_levels}")
print()
play_game(game)


=== Starting Game ===
Session ID: 8704
Total number of questions: 15


--- Level 1 ---
Q: Which pharaoh is credited with the reunification of Egypt, thus marking the start of the Middle Kingdom?

  [0] Mentuhotep II
  [1] Neferhotep I
  [2] Sobekneferu
  [3] Amenemhet III

Time remaining: 29.9s
 WRONG ANSWER!

 Game Over!
 Final earnings: $0.00

=== Game Summary ===
Reached Level: 1
Total Earnings: $0.00


In [10]:
# Show leaderboard position
lb = client.leaderboard.get(competition_id=comp_id, limit=10)
print(f"\n=== Leaderboard for {lb.competition.name} ===")
for i, entry in enumerate(lb.entries[:5], 1):
    marker = " <-- YOU" if entry.username == username else ""
    print(f"  {i}. {entry.username}: ${entry.score:,.2f} (Level {entry.reached_level}){marker}")


=== Leaderboard for Ancient History and Politics ===
  1. AleAssini: $1,024,000.00 (Level 15)
  2. supreme_leader: $1,024,000.00 (Level 15)
  3. luca_bordin: $1,024,000.00 (Level 15)
  4. Anonymous: $1,024,000.00 (Level 15)
  5. Jasmin: $1,024,000.00 (Level 15)


TODO

In [85]:
import json
from urllib.parse import quote_plus
from urllib.request import Request, urlopen


def fetch_wikipedia_raw_document(query: str):
    search_url = (
        "https://en.wikipedia.org/w/api.php"
        "?action=query"
        "&list=search"
        f"&srsearch={quote_plus(query)}"
        "&srlimit=1"
        "&format=json"
        "&utf8=1"
        "&redirects=1"
    )
    with urlopen(Request(search_url, headers={"User-Agent": "Mozilla/5.0"})) as response:
        search_data = json.load(response)

    search_results = search_data.get("query", {}).get("search", [])
    if not search_results:
        raise ValueError(f"No Wikipedia page found for query: {query}")

    title = search_results[0]["title"]
    raw_url = (
        "https://en.wikipedia.org/w/api.php"
        "?action=query"
        "&prop=revisions"
        "&rvprop=content"
        "&rvslots=main"
        f"&titles={quote_plus(title)}"
        "&formatversion=2"
        "&format=json"
        "&utf8=1"
        "&redirects=1"
    )
    with urlopen(Request(raw_url, headers={"User-Agent": "Mozilla/5.0"})) as response:
        page_data = json.load(response)

    page = page_data["query"]["pages"][0]
    revisions = page.get("revisions", [])
    if not revisions:
        raise ValueError(f"No raw content available for Wikipedia page: {title}")

    raw_document = revisions[0]["slots"]["main"]["content"]
    return title, raw_document


current_game = globals().get("game")
if current_game is None or current_game.current_question is None:
    current_game = client.game.start(competition_id=comp_id)
    game = current_game

current_question_text = current_game.current_question.text if current_game.current_question else ""
if not current_question_text:
    print("No active question available.")
else:
    wiki_title, wiki_raw_document = fetch_wikipedia_raw_document(current_question_text)
    print(f"Session ID: {current_game.session_id}")
    print(f"Question: {current_question_text}")
    print(f"Wikipedia title: {wiki_title}")
    print(wiki_raw_document)
    wiki_raw_path = "/content/gdrive/MyDrive/NLP_assignment/wiki_raw_document.txt"
    with open(wiki_raw_path, "w", encoding="utf-8") as wiki_raw_file:
        wiki_raw_file.write(wiki_raw_document)
    print(f"Saved raw Wikipedia document to: {wiki_raw_path}")


Session ID: 8705
Question: Which term describes Julius Caesar's famous statement 'veni, vidi, vici'?
Wikipedia title: Julius Caesar
{{Short description|Roman general and dictator (100–44 BC)}}
{{Redirect2|Gaius Julius Caesar|Caesar|the name|Gaius Julius Caesar (name)|text=For other uses, see [[Gaius Julius Caesar (disambiguation)]], [[Caesar (disambiguation)]], [[Julius Caesar (disambiguation)]], and [[Caesar (title)]]}}
{{pp|small=yes}}
{{Use British English|date=November 2024}}
{{Use dmy dates|date=December 2025}}
{{Infobox person
| name               = Julius Caesar
| image              = Retrato de Julio César (26724093101) (cropped).jpg
| image_upright      = 
| alt                = The Tusculum portrait, a marble sculpture of Julius Caesar
| caption            = The [[Tusculum portrait]], the only extant contemporary sculpture of Caesar 
| birth_date         = 12 or 13 July 100 BC<ref>{{harvnb|Badian|2009|p= [https://books.google.com/books?id=gzOXLGbIIYwC&pg=PA16 16]|ps=. All anc

In [86]:
import re


def clean_wikipedia_raw_text(raw_text: str) -> str:
    text = raw_text
    text = re.sub(r"<ref[^>/]*?>.*?</ref>", " ", text, flags=re.IGNORECASE | re.DOTALL)
    text = re.sub(r"<ref[^>]*/>", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"<!--.*?-->", " ", text, flags=re.DOTALL)
    text = re.sub(r"\{\{[^{}]*\}\}", " ", text)
    text = re.sub(r"\[\[(?:File|Image):[^\]]*\]\]", " ", text, flags=re.IGNORECASE)

    def replace_link(match: re.Match) -> str:
        target = match.group(1).strip()
        label = match.group(2)
        if label:
            return label.strip()
        if ":" in target:
            return " "
        return target.split("|")[-1].strip()

    text = re.sub(r"\[\[(.+?)(?:\|(.+?))?\]\]", replace_link, text)
    text = text.replace("[[", " ").replace("]]", " ")
    text = re.sub(r"'''''(.*?)'''''", r"\1", text)
    text = re.sub(r"'''(.*?)'''", r"\1", text)
    text = re.sub(r"''(.*?)''", r"\1", text)
    text = re.sub(r"^=+\s*(.*?)\s*=+$", r"\1", text, flags=re.MULTILINE)
    text = re.sub(r"^\s*[\*#;:]+\s*", "", text, flags=re.MULTILINE)
    text = re.sub(r"\{\|.*?\|\}", " ", text, flags=re.DOTALL)
    text = text.replace("{|", " ").replace("|}", " ")
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    return text.strip()


clean_wiki_text = clean_wikipedia_raw_text(wiki_raw_document)
clean_wiki_path = "/content/gdrive/MyDrive/NLP_assignment/wiki_clean_document.txt"
with open(clean_wiki_path, "w", encoding="utf-8") as clean_wiki_file:
    clean_wiki_file.write(clean_wiki_text)

print(clean_wiki_text[:4000])
print()
print(f"Saved cleaned Wikipedia text to: {clean_wiki_path}")


{{Infobox person
| name = Julius Caesar
| image = Retrato de Julio César (26724093101) (cropped).jpg
| image_upright =
| alt = The Tusculum portrait, a marble sculpture of Julius Caesar
| caption = The Tusculum portrait, the only extant contemporary sculpture of Caesar
| birth_date = 12 or 13 July 100 BC
| birth_place = Suburra, Rome
| death_date = 15 March 44 BC (aged 55)
| death_place = Theatre of Pompey, Rome
| death_cause = Assassination (stab wounds)
| occupation =
| office =
| notable_works = {{ubl| | }}
| spouse = {{Aligned table
| class= |fullwidth=on |leftright=on
| style=line-height:1.2em; |col2style=font-size:90%;
| Cossutia (disputed) |
| Cornelia | 84 BC; 69 BC
| Pompeia | 67 BC; 61 BC
| Calpurnia | 59 BC
}}
| partner = Cleopatra VII (mistress)
| children =
| parents =
| awards = Civic Crown
| module = {{Infobox military person | embed = yes
| allegiance = Roman Republic
| branch = Roman Army
| commands = XIII Legion
| battles =
Siege of Mytilene
Gallic Wars
Invasions of B

In [87]:
import math
import re
import json

CHUNK_SIZE = 1400
CHUNK_OVERLAP = 250
TOP_N_CHUNKS = 5


def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP):
    cleaned_text = re.sub(r"\s+", " ", text).strip()
    if not cleaned_text:
        return []

    step = max(chunk_size - overlap, 1)
    chunks = []
    for start in range(0, len(cleaned_text), step):
        chunk = cleaned_text[start:start + chunk_size].strip()
        if chunk:
            chunks.append(chunk)
        if start + chunk_size >= len(cleaned_text):
            break
    return chunks


def compute_option_support_scores(chunk: str, options: list[str]) -> dict:
    """Compute how well each option is supported by this chunk."""
    chunk_lower = chunk.lower()
    chunk_terms = set(re.findall(r"[a-z0-9]+", chunk_lower))
    
    scores = {}
    for i, option in enumerate(options):
        option_terms = set(re.findall(r"[a-z0-9]+", option.lower()))
        # Unique word matches (not repeated)
        unique_hits = len(chunk_terms & option_terms)
        # Bonus for dense option matching
        option_density = unique_hits / (len(option_terms) + 1)
        scores[["A", "B", "C", "D"][i]] = unique_hits + option_density
    
    return scores


def rank_chunk(chunk: str, question_text: str, options: list[str]) -> float:
    """Enhanced ranking that weighs question terms heavily and considers option support."""
    chunk_terms = set(re.findall(r"[a-z0-9]+", chunk.lower()))
    query_terms = set(re.findall(r"[a-z0-9]+", question_text.lower()))
    
    # Strong weight on question relevance
    query_score = len(chunk_terms & query_terms) * 3.0
    
    # Compute per-option support
    option_scores = compute_option_support_scores(chunk, options)
    max_option_score = max(option_scores.values())
    
    # Length bonus (prefer substantial chunks)
    length_bonus = min(len(chunk) / 1000.0, 1.0)
    
    return query_score + max_option_score + length_bonus


if not clean_wiki_text:
    raise ValueError("Clean Wikipedia text is empty. Run the cleaning cell first.")

current_question = current_game.current_question if current_game else None
if current_question is None:
    raise ValueError("No active question available for chunking.")

current_question_text = current_question.text
question_options = [opt.text for opt in current_question.options]

all_chunks = chunk_text(clean_wiki_text)
if not all_chunks:
    raise ValueError("No chunks were created from the cleaned Wikipedia text.")

all_ranked_chunks = [
    {
        "rank": idx + 1,
        "score": rank_chunk(chunk, current_question_text, question_options),
        "chunk": chunk,
        "option_scores": compute_option_support_scores(chunk, question_options),
    }
    for idx, chunk in enumerate(all_chunks)
]
all_ranked_chunks.sort(key=lambda item: item["score"], reverse=True)

selected_chunks = all_ranked_chunks[:TOP_N_CHUNKS]
selected_chunk_texts = [item["chunk"] for item in selected_chunks]
chunks_block = "\n\n".join(f"CHUNK {i+1}: {chunk}" for i, chunk in enumerate(selected_chunk_texts))

print(f"Created {len(all_chunks)} chunks from the cleaned Wikipedia text.")
print(f"Selected top {len(selected_chunk_texts)} chunks for scoring.")
print(f"\nTop chunks and their per-option support scores:")

# Compute and display per-chunk per-option confidences
selected_chunk_confidences = []
for i, chunk in enumerate(selected_chunk_texts):
    option_scores = compute_option_support_scores(chunk, question_options)
    total = sum(option_scores.values()) or 1.0
    confs = {label: score / total for label, score in option_scores.items()}
    selected_chunk_confidences.append({
        "chunk_index": i + 1,
        "score": selected_chunks[i]["score"],
        "chunk": chunk[:200],
        "confidences": confs,
    })
    print(f"\nChunk {i+1} (rank score {selected_chunks[i]['score']:.2f}):")
    print(f"  Option scores: {option_scores}")
    print(f"  Normalized: {confs}")

# Keep variables available for downstream cells
all_ranked_chunks = all_ranked_chunks
selected_chunks = selected_chunks
selected_chunk_texts = selected_chunk_texts
selected_chunk_confidences = selected_chunk_confidences
chunks_block = chunks_block

Created 65 chunks from the cleaned Wikipedia text.
Selected top 5 chunks for scoring.

Top chunks and their per-option support scores:

Chunk 1 (rank score 20.25):
  Option scores: {'A': 1.25, 'B': 1.25, 'C': 1.25, 'D': 1.25}
  Normalized: {'A': 0.25, 'B': 0.25, 'C': 0.25, 'D': 0.25}

Chunk 2 (rank score 17.25):
  Option scores: {'A': 1.25, 'B': 1.25, 'C': 1.25, 'D': 1.25}
  Normalized: {'A': 0.25, 'B': 0.25, 'C': 0.25, 'D': 0.25}

Chunk 3 (rank score 15.50):
  Option scores: {'A': 1.25, 'B': 1.25, 'C': 2.5, 'D': 1.25}
  Normalized: {'A': 0.2, 'B': 0.2, 'C': 0.4, 'D': 0.2}

Chunk 4 (rank score 15.50):
  Option scores: {'A': 1.25, 'B': 2.5, 'C': 2.5, 'D': 2.5}
  Normalized: {'A': 0.14285714285714285, 'B': 0.2857142857142857, 'C': 0.2857142857142857, 'D': 0.2857142857142857}

Chunk 5 (rank score 15.50):
  Option scores: {'A': 1.25, 'B': 1.25, 'C': 2.5, 'D': 1.25}
  Normalized: {'A': 0.2, 'B': 0.2, 'C': 0.4, 'D': 0.2}


In [90]:
import json
import os
import re
import torch

OUTPUT_PATH = "/content/gdrive/MyDrive/NLP_assignment/llm_confidences.json"
#MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

# Compute answer from chunk-based support scores (most reliable method)
print("Computing answer from chunk support scores...")
chunk_option_scores = {"A": 0.0, "B": 0.0, "C": 0.0, "D": 0.0}

for chunk_conf in selected_chunk_confidences:
    for label in ["A", "B", "C", "D"]:
        chunk_option_scores[label] += chunk_conf["confidences"][label]

# Average across chunks
num_chunks = len(selected_chunk_confidences)
if num_chunks > 0:
    chunk_option_scores = {label: score / num_chunks for label, score in chunk_option_scores.items()}

print(f"Chunk-based average scores: {chunk_option_scores}")

# Pick the option with highest chunk support
best_option = max(chunk_option_scores, key=chunk_option_scores.get)
best_score = chunk_option_scores[best_option]
print(f"Best supported option from chunks: {best_option} (score: {best_score:.3f})")

# Create final confidence: 1.0 for best option, 0 for others
final_conf = {"A": 0.0, "B": 0.0, "C": 0.0, "D": 0.0}
final_conf[best_option] = 1.0

question = current_game.current_question
if question is None:
    raise ValueError("No active question available.")

question_text = question.text
options = [opt.text for opt in question.options]

raw_details = {
    label: {"option": options[i], "confidence": final_conf[label]}
    for i, label in enumerate(["A", "B", "C", "D"])
}

with open(OUTPUT_PATH, "w", encoding="utf-8") as out_f:
    json.dump({
        "question": question_text,
        "options": options,
        "confidences": final_conf,
        "raw_model_output": f"Chunk-based scoring: {chunk_option_scores}",
        "model_used": "chunk_based_scoring"
    }, out_f, ensure_ascii=False, indent=2)

print("Used chunk-based scoring (most reliable).")
print("Saved LLM confidences to:", OUTPUT_PATH)
print(json.dumps(final_conf))

Computing answer from chunk support scores...
Chunk-based average scores: {'A': 0.20857142857142855, 'B': 0.23714285714285716, 'C': 0.31714285714285717, 'D': 0.23714285714285716}
Best supported option from chunks: C (score: 0.317)
Used chunk-based scoring (most reliable).
Saved LLM confidences to: /content/gdrive/MyDrive/NLP_assignment/llm_confidences.json
{"A": 0.0, "B": 0.0, "C": 1.0, "D": 0.0}


In [63]:
print(output)

[{'generated_text': "The primary reason for the expansion of the Macedonian kingdom in the 4th century BC is the support of the chunks given in the answer options. The expansion of the Macedonian kingdom in the 4th century BC was supported by the illicit practice of silver mining in the eastern territories, the legacy of the illicit practice of Greek culture, and the legacy of the expansion of Greek culture to the Near East during and after Alexander's conquests. The expansion of the kingdom was also influenced by the spread of Greek culture and language cemented by"}]


In [89]:
print("=" * 80)
print("QUESTION:", question_text)
print("=" * 80)
for i, opt in enumerate(options):
    print(f"  {['A', 'B', 'C', 'D'][i]}: {opt}")
print("=" * 80)
print("\nSELECTED CHUNKS FOR SCORING:")
for i, chunk in enumerate(selected_chunk_texts):
    print(f"\n--- CHUNK {i+1} ---")
    print(chunk[:400])
    print("...")

QUESTION: Which term describes Julius Caesar's famous statement 'veni, vidi, vici'?
  A: A diplomatic message
  B: A military strategy
  C: A victory declaration
  D: A battle cry

SELECTED CHUNKS FOR SCORING:

--- CHUNK 1 ---
thus, and the like) in many regional pronunciations, including the French one, from which the modern English pronunciation is derived. Caesar's cognomen itself became a title; it was promulgated by the Bible, which contains the famous verse "Render unto Caesar the things which are Caesar's, and unto God the things that are God's". The title became, from the late first millennium, Kaiser in German 
...

--- CHUNK 2 ---
oural manifestations headaches, vertigo, falls (possibly caused by muscle weakness due to nerve damage), sensory deficit, giddiness and insensibility and syncopal episodes were the results of cerebrovascular episodes, not epilepsy. Pliny the Elder reports in his Natural History that Caesar's father and forefather died without apparent cause while pu

START

In [65]:
system_prompt = """You are a helpful assistant.
give me only one number 0, 1, 2 or 3 corresponding to the correct option A, B, C or D respectively."""

user_prompt = f"""Check which answer options are supported by the chunks below.
Use only the chunks and give me the correct answer. dont think beyond them.

Question: {question_text}

Options:
A: {options[0]}
B: {options[1]}
C: {options[2]}
D: {options[3]}

Chunks:
{chunks_block}"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]

In [66]:
print(messages)

[{'role': 'system', 'content': 'You are a helpful assistant.\ngive me only one number 0, 1, 2 or 3 corresponding to the correct option A, B, C or D respectively.'}, {'role': 'user', 'content': 'Check which answer options are supported by the chunks below.\nUse only the chunks and give me the correct answer. dont think beyond them.\n\nQuestion: What was the primary reason for the expansion of the Macedonian kingdom in the 4th century BC?\n\nOptions:\nA: To gain access to rich mineral resources in the eastern territories.\nB: To conquer and subjugate neighboring barbarian tribes.\nC: The desire to spread Greek culture.\nD: To establish trade routes with the Persian Empire.\n\nChunks:\nCHUNK 1: ins between 167 and 148 BC (i.e. just before the establishment of the Roman province of Macedonia), and when the Romans lifted the ban on Macedonian silver mining in 158 BC it may simply have reflected the local reality of this illicit practice continuing regardless of the Senate\'s decree. Legacy 

In [67]:
print(user_prompt)

Check which answer options are supported by the chunks below.
Use only the chunks and give me the correct answer. dont think beyond them.

Question: What was the primary reason for the expansion of the Macedonian kingdom in the 4th century BC?

Options:
A: To gain access to rich mineral resources in the eastern territories.
B: To conquer and subjugate neighboring barbarian tribes.
C: The desire to spread Greek culture.
D: To establish trade routes with the Persian Empire.

Chunks:
CHUNK 1: ins between 167 and 148 BC (i.e. just before the establishment of the Roman province of Macedonia), and when the Romans lifted the ban on Macedonian silver mining in 158 BC it may simply have reflected the local reality of this illicit practice continuing regardless of the Senate's decree. Legacy The reigns of Philip II and Alexander the Great witnessed the demise of Classical Greece and the birth of Hellenistic civilization, following the spread of Greek culture to the Near East during and after Ale

In [ ]:
from huggingface_hub import login

login("")

In [52]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_id = "meta-llama/Llama-3.2-3B-Instruct"
# or: "meta-llama/Llama-3.2-3B" for the base, non-chat model

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,   # use torch.float16 if your GPU doesn't support bfloat16
    device_map="auto"
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

messages = [
    {"role": "user", "content": "Explain what a transformer is in simple terms."}
]

output = pipe(
    messages,
    max_new_tokens=256,
    temperature=0.7,
    do_sample=True
)

print(output[0]["generated_text"][-1]["content"])

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A transformer is a device that helps change the voltage (or strength) of electricity. Think of it like a special kind of elevator for electricity.

Imagine you have a battery that can provide a small amount of electricity (like 9 volts). But you need to charge your phone, which requires more electricity (like 12 volts). That's where the transformer comes in.

The transformer takes the 9-volt electricity from the battery and "steps up" to 12 volts, making it strong enough to power your phone. It does this by using a special coil of wire inside the transformer that changes the voltage.

The transformer can also "step down" voltage, which means it can take high voltage electricity (like 120 volts) and make it safe for use in homes (like 12 volts). This helps protect people and devices from getting hurt by strong electricity.

In simple terms, a transformer is a device that helps change the voltage of electricity to make it safe and strong enough for use in different devices.


In [78]:


output = pipe(
    messages,
    max_new_tokens=512,
    temperature=1,
    do_sample=False
)

print(output[0]["generated_text"][-1]["content"])

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


2


In [76]:
print(output)

[{'generated_text': [{'role': 'system', 'content': 'You are a helpful assistant.\ngive me only one number 0, 1, 2 or 3 corresponding to the correct option A, B, C or D respectively.'}, {'role': 'user', 'content': 'Check which answer options are supported by the chunks below.\nUse only the chunks and give me the correct answer. dont think beyond them.\n\nQuestion: What was the primary reason for the expansion of the Macedonian kingdom in the 4th century BC?\n\nOptions:\nA: To gain access to rich mineral resources in the eastern territories.\nB: To conquer and subjugate neighboring barbarian tribes.\nC: The desire to spread Greek culture.\nD: To establish trade routes with the Persian Empire.\n\nChunks:\nCHUNK 1: ins between 167 and 148 BC (i.e. just before the establishment of the Roman province of Macedonia), and when the Romans lifted the ban on Macedonian silver mining in 158 BC it may simply have reflected the local reality of this illicit practice continuing regardless of the Senat

# AUTOMATED PIPELINE

In [111]:
import os
import json
from datetime import datetime
from urllib.parse import quote_plus
from urllib.request import Request, urlopen
import re

# Create base output directory for all questions
BASE_OUTPUT_DIR = "/content/gdrive/MyDrive/NLP_assignment/automated_pipeline"
os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)

STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "been", "but", "by", "for", "from",
    "has", "have", "how", "in", "is", "it", "its", "of", "on", "or", "that", "the",
    "their", "there", "these", "this", "those", "to", "was", "were", "what", "when",
    "where", "which", "who", "why", "with", "during", "did", "do", "does", "according",
    "following", "among", "between", "into", "than", "then", "through", "under", "over",
    "after", "before", "period", "status", "purpose", "reason", "used", "use"
}


def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", text.lower()).strip()


def extract_keywords(text: str) -> list[str]:
    words = re.findall(r"[a-z0-9]+", text.lower())
    keywords = [word for word in words if len(word) > 2 and word not in STOPWORDS]
    seen = set()
    ordered_keywords = []
    for word in keywords:
        if word not in seen:
            ordered_keywords.append(word)
            seen.add(word)
    return ordered_keywords


def build_search_queries(question_text: str, options: list[str]) -> list[str]:
    capital_phrases = re.findall(r"(?:[A-Z][a-z0-9]+(?:\s+[A-Z][a-z0-9]+)*)", question_text)
    important_terms = extract_keywords(question_text)
    option_terms = extract_keywords(" ".join(options))

    queries = []
    if capital_phrases:
        queries.append(" ".join(capital_phrases[:4]))
    if important_terms:
        queries.append(" ".join(important_terms[:10]))
    if important_terms[:4]:
        queries.append(" ".join(important_terms[:4]))
    if option_terms[:3] and important_terms[:6]:
        queries.append(" ".join(important_terms[:6] + option_terms[:2]))
    queries.append(question_text)

    deduped = []
    seen = set()
    for query in queries:
        cleaned = normalize_text(query)
        if cleaned and cleaned not in seen:
            deduped.append(query)
            seen.add(cleaned)
    return deduped


def fetch_wikipedia_raw_document(query: str):
    search_url = (
        "https://en.wikipedia.org/w/api.php"
        "?action=query"
        "&list=search"
        f"&srsearch={quote_plus(query)}"
        "&srlimit=5"
        "&format=json"
        "&utf8=1"
        "&redirects=1"
    )
    with urlopen(Request(search_url, headers={"User-Agent": "Mozilla/5.0"})) as response:
        search_data = json.load(response)

    search_results = search_data.get("query", {}).get("search", [])
    if not search_results:
        raise ValueError(f"No Wikipedia page found for query: {query}")

    query_terms = set(extract_keywords(query))
    best_result = None
    best_overlap = -1
    for result in search_results:
        title_terms = set(extract_keywords(result.get("title", "")))
        overlap = len(query_terms & title_terms)
        if overlap > best_overlap:
            best_overlap = overlap
            best_result = result

    title = best_result["title"] if best_result else search_results[0]["title"]
    raw_url = (
        "https://en.wikipedia.org/w/api.php"
        "?action=query"
        "&prop=revisions"
        "&rvprop=content"
        "&rvslots=main"
        f"&titles={quote_plus(title)}"
        "&formatversion=2"
        "&format=json"
        "&utf8=1"
        "&redirects=1"
    )
    with urlopen(Request(raw_url, headers={"User-Agent": "Mozilla/5.0"})) as response:
        page_data = json.load(response)

    page = page_data["query"]["pages"][0]
    revisions = page.get("revisions", [])
    if not revisions:
        raise ValueError(f"No raw content available for Wikipedia page: {title}")

    raw_document = revisions[0]["slots"]["main"]["content"]
    return title, raw_document


print("=" * 80)
print("STARTING AUTOMATED PIPELINE")
print("=" * 80)

current_game = client.game.start(competition_id=comp_id)
print(f"Game started. Session ID: {current_game.session_id}")
print(f"Total questions: {current_game.state.competition.max_levels}")
print()

question_num = 0
total_correct = 0

while current_game.in_progress:
    question_num += 1
    question = current_game.current_question

    if not question:
        print("No question available. Game ended.")
        break

    print(f"\n{'=' * 80}")
    print(f"QUESTION {question_num} | Level {current_game.current_level}")
    print(f"{'=' * 80}")

    question_text = question.text
    options = [opt.text for opt in question.options]

    print(f"Q: {question_text}")
    print("\nOptions:")
    for i, opt in enumerate(options):
        print(f"  [{i}] {opt}")

    print("\n[1/5] Fetching Wikipedia document...")
    wiki_raw_document = ""
    wiki_title = "N/A"
    candidate_queries = build_search_queries(question_text, options)
    last_error = None

    for candidate_query in candidate_queries:
        try:
            print(f"  -> trying: {candidate_query}")
            wiki_title, wiki_raw_document = fetch_wikipedia_raw_document(candidate_query)
            if wiki_raw_document:
                break
        except Exception as e:
            last_error = e

    if not wiki_raw_document:
        print(f"  ❌ No usable Wikipedia page found. Using empty chunks. Last error: {last_error}")
        wiki_title = "N/A"

    print(f"  ✓ Fetched: {wiki_title} ({len(wiki_raw_document)} chars)")

    print("[2/5] Cleaning Wikipedia text...")
    def clean_wikipedia_raw_text(raw_text: str) -> str:
        text = raw_text
        text = re.sub(r"<ref[^>/]*?>.*?</ref>", " ", text, flags=re.IGNORECASE | re.DOTALL)
        text = re.sub(r"<ref[^>]*/?>", " ", text, flags=re.IGNORECASE)
        text = re.sub(r"<!--.*?-->", " ", text, flags=re.DOTALL)
        text = re.sub(r"\{\{[^{}]*\}\}", " ", text)
        text = re.sub(r"\[\[(?:File|Image):[^\]]*\]\]", " ", text, flags=re.IGNORECASE)

        def replace_link(match):
            target = match.group(1).strip()
            label = match.group(2)
            if label:
                return label.strip()
            if ":" in target:
                return " "
            return target.split("|")[-1].strip()

        text = re.sub(r"\[\[(.+?)(?:\|(.+?))?\]\]", replace_link, text)
        text = text.replace("[[", " ").replace("]]", " ")
        text = re.sub(r"'''''(.*?)'''''", r"\1", text)
        text = re.sub(r"'''(.*?)'''", r"\1", text)
        text = re.sub(r"''(.*?)''", r"\1", text)
        text = re.sub(r"^=+\s*(.*?)\s*=+$", r"\1", text, flags=re.MULTILINE)
        text = re.sub(r"^\s*[\*#;:]+\s*", "", text, flags=re.MULTILINE)
        text = re.sub(r"\{\|.*?\|\}", " ", text, flags=re.DOTALL)
        text = text.replace("{|", " ").replace("|}", " ")
        text = re.sub(r"\n{3,}", "\n\n", text)
        text = re.sub(r"[ \t]+", " ", text)
        text = re.sub(r" *\n *", "\n", text)
        return text.strip()

    clean_wiki_text = clean_wikipedia_raw_text(wiki_raw_document)
    print(f"  ✓ Cleaned text ({len(clean_wiki_text)} chars)")

    print("[3/5] Chunking and ranking...")
    CHUNK_SIZE = 4000
    CHUNK_OVERLAP = 700
    TOP_N_CHUNKS = 15

    def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
        cleaned_text = re.sub(r"\s+", " ", text).strip()
        if not cleaned_text:
            return []
        step = max(chunk_size - overlap, 1)
        chunks = []
        for start in range(0, len(cleaned_text), step):
            chunk = cleaned_text[start:start + chunk_size].strip()
            if chunk:
                chunks.append(chunk)
            if start + chunk_size >= len(cleaned_text):
                break
        return chunks

    def compute_option_support_scores(chunk, options):
        chunk_terms = set(re.findall(r"[a-z0-9]+", chunk.lower()))
        scores = {}
        for i, option in enumerate(options):
            option_terms = set(re.findall(r"[a-z0-9]+", option.lower()))
            unique_hits = len(chunk_terms & option_terms)
            option_density = unique_hits / (len(option_terms) + 1)
            scores[["A", "B", "C", "D"][i]] = unique_hits + option_density
        return scores

    def rank_chunk_score(chunk, question_text, options):
        chunk_terms = set(re.findall(r"[a-z0-9]+", chunk.lower()))
        query_keywords = extract_keywords(question_text)
        query_score = len(chunk_terms & set(query_keywords)) * 4.0
        option_scores = compute_option_support_scores(chunk, options)
        max_option_score = max(option_scores.values())
        length_bonus = min(len(chunk) / 2500.0, 1.0)
        exact_phrase_bonus = 2.0 if any(term in chunk.lower() for term in query_keywords[:4]) else 0.0
        return query_score + max_option_score + length_bonus + exact_phrase_bonus

    all_chunks = chunk_text(clean_wiki_text)
    if not all_chunks:
        all_chunks = ["[No content available]"]

    all_ranked_chunks = [
        {
            "rank": idx + 1,
            "score": rank_chunk_score(chunk, question_text, options),
            "chunk": chunk,
            "option_scores": compute_option_support_scores(chunk, options),
        }
        for idx, chunk in enumerate(all_chunks)
    ]
    all_ranked_chunks.sort(key=lambda item: item["score"], reverse=True)

    selected_chunks = all_ranked_chunks[:TOP_N_CHUNKS]
    selected_chunk_texts = [item["chunk"] for item in selected_chunks]

    print(f"  ✓ Created {len(all_chunks)} chunks, selected top {len(selected_chunk_texts)}")

    print("[4/5] Weighted hard voting from chunk winners...")
    chunk_vote_counts = {"A": 0, "B": 0, "C": 0, "D": 0}
    chunk_vote_confidence_sum = {"A": 0.0, "B": 0.0, "C": 0.0, "D": 0.0}
    selected_chunk_confidences = []
    chunk_vote_details = []

    for i, selected_chunk in enumerate(selected_chunks):
        option_scores = selected_chunk["option_scores"]
        total = sum(option_scores.values()) or 1.0
        confs = {label: score / total for label, score in option_scores.items()}
        selected_chunk_confidences.append({
            "chunk_index": i + 1,
            "score": selected_chunk["score"],
            "chunk": selected_chunk["chunk"][:200],
            "confidences": confs,
        })

        ranked_confidences = sorted(confs.items(), key=lambda item: item[1], reverse=True)
        chunk_winner, chunk_winner_conf = ranked_confidences[0]
        runner_up_conf = ranked_confidences[1][1] if len(ranked_confidences) > 1 else 0.0
        vote_weight = max(chunk_winner_conf - runner_up_conf, 0.0)

        if vote_weight >= 0.05:
            chunk_vote_counts[chunk_winner] += 1
            chunk_vote_confidence_sum[chunk_winner] += vote_weight
        chunk_vote_details.append({
            "chunk_index": i + 1,
            "winner": chunk_winner,
            "winner_confidence": chunk_winner_conf,
            "runner_up_confidence": runner_up_conf,
            "vote_weight": vote_weight,
            "confidences": confs,
        })

        print(f"\nChunk {i+1} (rank score {selected_chunk['score']:.2f}):")
        print(f"  Option scores: {option_scores}")
        print(f"  Normalized: {confs}")
        print(f"  Vote: {chunk_winner} ({chunk_winner_conf:.3f}), weight={vote_weight:.3f}")

    vote_fraction_scores = {
        label: chunk_vote_counts[label] / max(len(selected_chunks), 1)
        for label in ["A", "B", "C", "D"]
    }
    print(f"\n  Vote counts: {chunk_vote_counts}")
    print(f"  Vote fractions: {vote_fraction_scores}")
    print(f"  Weight sums: {chunk_vote_confidence_sum}")

    best_option_letter = max(
        ["A", "B", "C", "D"],
        key=lambda label: (
            chunk_vote_counts[label],
            chunk_vote_confidence_sum[label],
            vote_fraction_scores[label],
        ),
    )
    best_option_idx = ["A", "B", "C", "D"].index(best_option_letter)
    chunk_option_scores = vote_fraction_scores

    print(f"  ✓ Weighted hard-vote answer: {best_option_idx} ({best_option_letter})")

    print("[5/5] Submitting answer...")
    result = current_game.answer(best_option_idx)

    is_correct = result.correct
    if is_correct:
        total_correct += 1
        print(f"  ✓ CORRECT! Earned: ${result.earned_amount:,.2f}")
    else:
        print("  ✗ WRONG!")

    question_dir = os.path.join(BASE_OUTPUT_DIR, f"Q{question_num:02d}_{best_option_letter}")
    os.makedirs(question_dir, exist_ok=True)

    doc = {
        "question_number": question_num,
        "level": current_game.current_level,
        "question": question_text,
        "options": {label: options[i] for i, label in enumerate(["A", "B", "C", "D"])},
        "wiki_title": wiki_title,
        "wiki_queries_tried": candidate_queries,
        "selected_chunks": selected_chunk_texts,
        "chunk_vote_details": chunk_vote_details,
        "chunk_vote_counts": chunk_vote_counts,
        "chunk_vote_confidence_sum": chunk_vote_confidence_sum,
        "vote_fraction_scores": vote_fraction_scores,
        "predicted_answer": best_option_letter,
        "predicted_answer_index": best_option_idx,
        "is_correct": is_correct,
        "earned": float(result.earned_amount),
        "timestamp": datetime.now().isoformat(),
    }

    with open(os.path.join(question_dir, "metadata.json"), "w", encoding="utf-8") as f:
        json.dump(doc, f, ensure_ascii=False, indent=2)

    with open(os.path.join(question_dir, "question.txt"), "w", encoding="utf-8") as f:
        f.write(f"Question {question_num} (Level {current_game.current_level}):\n\n")
        f.write(question_text + "\n\n")
        f.write("Options:\n")
        for i, opt in enumerate(options):
            mark = " <-- PREDICTED" if ["A", "B", "C", "D"][i] == best_option_letter else ""
            mark += " <-- CORRECT" if is_correct and ["A", "B", "C", "D"][i] == best_option_letter else ""
            f.write(f"  {i}: {opt}{mark}\n")

    with open(os.path.join(question_dir, "chunks.txt"), "w", encoding="utf-8") as f:
        f.write(f"Wikipedia source: {wiki_title}\n\n")
        for i, chunk in enumerate(selected_chunk_texts):
            f.write(f"=== CHUNK {i+1} ===\n{chunk}\n\n")

    print(f"  ✓ Saved to: {question_dir}")

    if result.game_over:
        print(f"\n{'=' * 80}")
        print("GAME OVER!")
        print(f"Total questions answered: {question_num}")
        print(f"Correct answers: {total_correct}")
        print(f"Accuracy: {100 * total_correct / question_num:.1f}%")
        print(f"Final earnings: ${result.earned_amount:,.2f}")
        print(f"Output directory: {BASE_OUTPUT_DIR}")
        print(f"{'=' * 80}\n")
        break

print("Automated pipeline completed!")

STARTING AUTOMATED PIPELINE
Game started. Session ID: 8813
Total questions: 15


QUESTION 1 | Level 1
Q: What term describes the period of Roman history from the traditional end of the Roman Republic in 27 BC until the abdication of Romulus Augustulus in AD 476?

Options:
  [0] The Roman Empire
  [1] The Roman Republic
  [2] The Dominate
  [3] The Principate

[1/5] Fetching Wikipedia document...
  -> trying: What Roman Roman Republic Romulus Augustulus
  ✓ Fetched: Roman people (109082 chars)
[2/5] Cleaning Wikipedia text...
  ✓ Cleaned text (70337 chars)
[3/5] Chunking and ranking...
  ✓ Created 22 chunks, selected top 15
[4/5] Weighted hard voting from chunk winners...

Chunk 1 (rank score 34.75):
  Option scores: {'A': 3.75, 'B': 2.5, 'C': 1.3333333333333333, 'D': 1.3333333333333333}
  Normalized: {'A': 0.42056074766355145, 'B': 0.28037383177570097, 'C': 0.14953271028037382, 'D': 0.14953271028037382}
  Vote: A (0.421), weight=0.140

Chunk 2 (rank score 30.75):
  Option scores: {'A':

In [114]:
# Semantic search over the cleaned document for each answer option.
# This version uses broader chunks plus both word and character n-gram similarity
# so short answer options do not collapse to all-zero scores.

import re
import json
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

if not clean_wiki_text:
    raise ValueError("clean_wiki_text is empty. Run the cleaning cell first.")

current_question = None
if "current_game" in globals() and current_game and current_game.current_question:
    current_question = current_game.current_question
elif "game" in globals() and game and game.current_question:
    current_question = game.current_question

question_text = current_question.text if current_question else globals().get("question_text", "")
options = [opt.text for opt in current_question.options] if current_question else globals().get("options", [])

if not question_text or not options:
    raise ValueError("No question/options available. Run a question cell first or keep the last question variables in memory.")

# Build larger semantic windows from the full cleaned document
paragraphs = [block.strip() for block in re.split(r"\n{2,}", clean_wiki_text) if len(block.strip()) > 80]
if not paragraphs:
    paragraphs = [clean_wiki_text]

# Add sentence-level windows as a fallback when paragraph-level context is too broad.
sentences = [sentence.strip() for sentence in re.split(r"(?<=[.!?])\s+|\n+", clean_wiki_text) if len(sentence.strip()) > 40]
if not sentences:
    sentences = [clean_wiki_text]

windows = paragraphs + sentences

# Expand each option with a few question keywords so the query has some context.
question_keywords = [word for word in re.findall(r"[a-z0-9]+", question_text.lower()) if len(word) > 2]
option_queries = []
for option_text in options:
    option_query = f"{option_text} {' '.join(question_keywords[:6])}".strip()
    option_queries.append(option_query)

# Combine word-level and character-level similarity scores.
word_vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1)
char_vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=1)

word_matrix = word_vectorizer.fit_transform(option_queries + windows)
char_matrix = char_vectorizer.fit_transform(option_queries + windows)

option_word = word_matrix[:len(options)]
window_word = word_matrix[len(options):]
option_char = char_matrix[:len(options)]
window_char = char_matrix[len(options):]

option_confidences = {}
option_best_hits = {}
option_match_details = {}

for option_index, option_text in enumerate(options):
    word_similarities = cosine_similarity(option_word[option_index:option_index + 1], window_word).flatten()
    char_similarities = cosine_similarity(option_char[option_index:option_index + 1], window_char).flatten()

    # Blend word and character similarity. Character similarity helps when wording is paraphrased.
    similarities = (0.45 * word_similarities) + (0.55 * char_similarities)
    best_window_index = int(similarities.argmax())
    best_similarity = float(similarities[best_window_index])

    top_indices = similarities.argsort()[::-1][:5]
    top_hits = [
        {
            "window": windows[idx],
            "similarity": float(similarities[idx]),
            "word_similarity": float(word_similarities[idx]),
            "char_similarity": float(char_similarities[idx]),
        }
        for idx in top_indices
    ]

    label = chr(ord("A") + option_index)
    option_confidences[label] = best_similarity
    option_best_hits[label] = {
        "best_window": windows[best_window_index],
        "best_similarity": best_similarity,
    }
    option_match_details[label] = top_hits

confidence_total = sum(option_confidences.values()) or 1.0
normalized_confidences = {
    label: score / confidence_total
    for label, score in option_confidences.items()
}

best_option = max(normalized_confidences, key=normalized_confidences.get)
best_option_idx = ["A", "B", "C", "D"].index(best_option)

print("=" * 80)
print("SEMANTIC SEARCH OPTION SCORES")
print("=" * 80)
print(f"Question: {question_text}")
print()
for label, option_text in zip(["A", "B", "C", "D"], options):
    print(f"{label}: {option_text}")
    print(f"  raw confidence: {option_confidences[label]:.6f}")
    print(f"  normalized confidence: {normalized_confidences[label]:.6f}")
    print(f"  best matching window: {option_best_hits[label]['best_window'][:220]}")
    print()

print(f"Best option from semantic search: {best_option} ({best_option_idx})")

semantic_output = {
    "question": question_text,
    "options": options,
    "raw_confidences": option_confidences,
    "normalized_confidences": normalized_confidences,
    "best_option": best_option,
    "best_option_index": best_option_idx,
    "best_hits": option_best_hits,
    "top_matches": option_match_details,
}

semantic_output_path = "/content/gdrive/MyDrive/NLP_assignment/semantic_search_confidences.json"
with open(semantic_output_path, "w", encoding="utf-8") as semantic_file:
    json.dump(semantic_output, semantic_file, ensure_ascii=False, indent=2)

print(f"Saved semantic search comparison to: {semantic_output_path}")

SEMANTIC SEARCH OPTION SCORES
Question: Which term describes Julius Caesar's famous statement 'veni, vidi, vici'?

A: A diplomatic message
  raw confidence: 0.050583
  normalized confidence: 0.229549
  best matching window: The relationship between Middle Egyptian and Late Egyptian has been described as being similar to that between Latin and Italian.

B: A military strategy
  raw confidence: 0.057989
  normalized confidence: 0.263156
  best matching window: The relationship between Middle Egyptian and Late Egyptian has been described as being similar to that between Latin and Italian.

C: A victory declaration
  raw confidence: 0.056588
  normalized confidence: 0.256803
  best matching window: The relationship between Middle Egyptian and Late Egyptian has been described as being similar to that between Latin and Italian.

D: A battle cry
  raw confidence: 0.055198
  normalized confidence: 0.250492
  best matching window: The relationship between Middle Egyptian and Late Egyptian has b

In [124]:
# Live game test for semantic search scoring
# Starts a fresh game and answers each question using the semantic search matcher.

import os
import json
import re
from datetime import datetime
from urllib.parse import quote_plus
from urllib.request import Request, urlopen
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

SEMANTIC_TEST_DIR = "/content/gdrive/MyDrive/NLP_assignment/semantic_game_test"
os.makedirs(SEMANTIC_TEST_DIR, exist_ok=True)

STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "been", "but", "by", "for", "from",
    "has", "have", "how", "in", "is", "it", "its", "of", "on", "or", "that", "the",
    "their", "there", "these", "this", "those", "to", "was", "were", "what", "when",
    "where", "which", "who", "why", "with", "during", "did", "do", "does", "according",
    "following", "among", "between", "into", "than", "then", "through", "under", "over",
    "after", "before", "period", "status", "purpose", "reason", "used", "use"
}


def extract_keywords(text: str) -> list[str]:
    words = re.findall(r"[a-z0-9]+", text.lower())
    keywords = [word for word in words if len(word) > 2 and word not in STOPWORDS]
    seen = set()
    ordered_keywords = []
    for word in keywords:
        if word not in seen:
            ordered_keywords.append(word)
            seen.add(word)
    return ordered_keywords


def fetch_wikipedia_raw_document(query: str):
    search_url = (
        "https://en.wikipedia.org/w/api.php"
        "?action=query"
        "&list=search"
        f"&srsearch={quote_plus(query)}"
        "&srlimit=5"
        "&format=json"
        "&utf8=1"
        "&redirects=1"
    )
    with urlopen(Request(search_url, headers={"User-Agent": "Mozilla/5.0"})) as response:
        search_data = json.load(response)

    search_results = search_data.get("query", {}).get("search", [])
    if not search_results:
        raise ValueError(f"No Wikipedia page found for query: {query}")

    query_terms = set(extract_keywords(query))
    best_result = None
    best_overlap = -1
    for result in search_results:
        title_terms = set(extract_keywords(result.get("title", "")))
        overlap = len(query_terms & title_terms)
        if overlap > best_overlap:
            best_overlap = overlap
            best_result = result

    title = best_result["title"] if best_result else search_results[0]["title"]
    raw_url = (
        "https://en.wikipedia.org/w/api.php"
        "?action=query"
        "&prop=revisions"
        "&rvprop=content"
        "&rvslots=main"
        f"&titles={quote_plus(title)}"
        "&formatversion=2"
        "&format=json"
        "&utf8=1"
        "&redirects=1"
    )
    with urlopen(Request(raw_url, headers={"User-Agent": "Mozilla/5.0"})) as response:
        page_data = json.load(response)

    page = page_data["query"]["pages"][0]
    revisions = page.get("revisions", [])
    if not revisions:
        raise ValueError(f"No raw content available for Wikipedia page: {title}")

    raw_document = revisions[0]["slots"]["main"]["content"]
    return title, raw_document


def clean_wikipedia_raw_text(raw_text: str) -> str:
    text = raw_text
    text = re.sub(r"<ref[^>/]*?>.*?</ref>", " ", text, flags=re.IGNORECASE | re.DOTALL)
    text = re.sub(r"<ref[^>]*/?>", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"<!--.*?-->", " ", text, flags=re.DOTALL)
    text = re.sub(r"\{\{[^{}]*\}\}", " ", text)
    text = re.sub(r"\[\[(?:File|Image):[^\]]*\]\]", " ", text, flags=re.IGNORECASE)

    def replace_link(match):
        target = match.group(1).strip()
        label = match.group(2)
        if label:
            return label.strip()
        if ":" in target:
            return " "
        return target.split("|")[-1].strip()

    text = re.sub(r"\[\[(.+?)(?:\|(.+?))?\]\]", replace_link, text)
    text = text.replace("[[", " ").replace("]]", " ")
    text = re.sub(r"'''''(.*?)'''''", r"\1", text)
    text = re.sub(r"'''(.*?)'''", r"\1", text)
    text = re.sub(r"''(.*?)''", r"\1", text)
    text = re.sub(r"^=+\s*(.*?)\s*=+$", r"\1", text, flags=re.MULTILINE)
    text = re.sub(r"^\s*[\*#;:]+\s*", "", text, flags=re.MULTILINE)
    text = re.sub(r"\{\|.*?\|\}", " ", text, flags=re.DOTALL)
    text = text.replace("{|", " ").replace("|}", " ")
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    return text.strip()


def semantic_option_scores(clean_text: str, question_text: str, options: list[str]):
    paragraphs = [block.strip() for block in re.split(r"\n{2,}", clean_text) if len(block.strip()) > 80]
    if not paragraphs:
        paragraphs = [clean_text]

    sentences = [sentence.strip() for sentence in re.split(r"(?<=[.!?])\s+|\n+", clean_text) if len(sentence.strip()) > 40]
    if not sentences:
        sentences = [clean_text]

    windows = paragraphs + sentences
    question_keywords = [word for word in re.findall(r"[a-z0-9]+", question_text.lower()) if len(word) > 2]
    option_queries = [f"{option_text} {' '.join(question_keywords[:6])}".strip() for option_text in options]

    word_vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1)
    char_vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=1)

    word_matrix = word_vectorizer.fit_transform(option_queries + windows)
    char_matrix = char_vectorizer.fit_transform(option_queries + windows)

    option_word = word_matrix[:len(options)]
    window_word = word_matrix[len(options):]
    option_char = char_matrix[:len(options)]
    window_char = char_matrix[len(options):]

    raw_scores = {}
    match_details = {}

    for option_index, option_text in enumerate(options):
        word_similarities = cosine_similarity(option_word[option_index:option_index + 1], window_word).flatten()
        char_similarities = cosine_similarity(option_char[option_index:option_index + 1], window_char).flatten()
        similarities = (0.45 * word_similarities) + (0.55 * char_similarities)
        best_window_index = int(similarities.argmax())
        best_similarity = float(similarities[best_window_index])
        label = chr(ord("A") + option_index)
        raw_scores[label] = best_similarity
        match_details[label] = {
            "best_window": windows[best_window_index],
            "best_similarity": best_similarity,
            "top_windows": [
                {
                    "window": windows[idx],
                    "similarity": float(similarities[idx]),
                    "word_similarity": float(word_similarities[idx]),
                    "char_similarity": float(char_similarities[idx]),
                }
                for idx in similarities.argsort()[::-1][:5]
            ],
        }

    total = sum(raw_scores.values()) or 1.0
    normalized = {label: score / total for label, score in raw_scores.items()}
    best_option = max(normalized, key=normalized.get)
    best_option_idx = ["A", "B", "C", "D"].index(best_option)
    return raw_scores, normalized, best_option, best_option_idx, match_details


def save_question_bundle(question_dir: str, payload: dict):
    os.makedirs(question_dir, exist_ok=True)
    with open(os.path.join(question_dir, "semantic_question.json"), "w", encoding="utf-8") as handle:
        json.dump(payload, handle, ensure_ascii=False, indent=2)


print("=" * 80)
print("STARTING SEMANTIC SEARCH GAME TEST")
print("=" * 80)

semantic_game = client.game.start(competition_id=comp_id)
print(f"Game started. Session ID: {semantic_game.session_id}")
print(f"Total questions: {semantic_game.state.competition.max_levels}")
print()

question_num = 0
total_correct = 0

while semantic_game.in_progress:
    question_num += 1
    question = semantic_game.current_question
    if not question:
        print("No question available. Game ended.")
        break

    question_text = question.text
    options = [opt.text for opt in question.options]

    print(f"\n{'=' * 80}")
    print(f"QUESTION {question_num} | Level {semantic_game.current_level}")
    print(f"Q: {question_text}")
    for i, opt in enumerate(options):
        print(f"  [{i}] {opt}")

    print("[1/4] Fetching Wikipedia document...")
    candidate_queries = []
    query_words = extract_keywords(question_text)
    if query_words:
        candidate_queries.append(" ".join(query_words[:8]))
    candidate_queries.append(question_text)

    wiki_title = "N/A"
    wiki_raw_document = ""
    last_error = None
    for candidate_query in candidate_queries:
        try:
            print(f"  -> trying: {candidate_query}")
            wiki_title, wiki_raw_document = fetch_wikipedia_raw_document(candidate_query)
            if wiki_raw_document:
                break
        except Exception as e:
            last_error = e

    if not wiki_raw_document:
        print(f"  ❌ No usable Wikipedia page found. Last error: {last_error}")
        clean_wiki_text = ""
    else:
        print(f"  ✓ Fetched: {wiki_title} ({len(wiki_raw_document)} chars)")
        print("[2/4] Cleaning Wikipedia text...")
        clean_wiki_text = clean_wikipedia_raw_text(wiki_raw_document)
        print(f"  ✓ Cleaned text ({len(clean_wiki_text)} chars)")

    print("[3/4] Scoring semantic matches...")
    if clean_wiki_text:
        raw_scores, normalized_scores, best_option, best_option_idx, match_details = semantic_option_scores(
            clean_wiki_text, question_text, options
        )
    else:
        raw_scores = {"A": 0.0, "B": 0.0, "C": 0.0, "D": 0.0}
        normalized_scores = {"A": 0.25, "B": 0.25, "C": 0.25, "D": 0.25}
        best_option = "A"
        best_option_idx = 0
        match_details = {}

    for label, option_text in zip(["A", "B", "C", "D"], options):
        print(f"  {label}: raw={raw_scores[label]:.6f}, norm={normalized_scores[label]:.6f} -> {option_text}")

    print(f"[4/4] Submitting answer {best_option_idx} ({best_option})...")
    result = semantic_game.answer(best_option_idx)
    if result.correct:
        total_correct += 1
        print(f"  ✓ CORRECT! Earned: ${result.earned_amount:,.2f}")
    else:
        print("  ✗ WRONG!")

    question_dir = os.path.join(SEMANTIC_TEST_DIR, f"Q{question_num:02d}_{best_option}")
    payload = {
        "question_number": question_num,
        "level": semantic_game.current_level,
        "question": question_text,
        "options": options,
        "wiki_title": wiki_title,
        "wiki_queries_tried": candidate_queries,
        "clean_wiki_length": len(clean_wiki_text),
        "raw_confidences": raw_scores,
        "normalized_confidences": normalized_scores,
        "best_option": best_option,
        "best_option_index": best_option_idx,
        "match_details": match_details,
        "correct": result.correct,
        "earned": float(result.earned_amount),
        "timestamp": datetime.now().isoformat(),
    }
    save_question_bundle(question_dir, payload)
    print(f"  ✓ Saved to: {question_dir}")

    if result.game_over:
        print(f"\n{'=' * 80}")
        print("GAME OVER!")
        print(f"Questions answered: {question_num}")
        print(f"Correct answers: {total_correct}")
        print(f"Accuracy: {100 * total_correct / question_num:.1f}%")
        print(f"Final earnings: ${result.earned_amount:,.2f}")
        print(f"Output directory: {SEMANTIC_TEST_DIR}")
        print(f"{'=' * 80}")
        break

print("Semantic search game test completed!")

STARTING SEMANTIC SEARCH GAME TEST
Game started. Session ID: 8839
Total questions: 15


QUESTION 1 | Level 1
Q: What is the fundamental principle of Xenia according to ancient Greek culture?
  [0] Hospitality and generosity towards strangers
  [1] Slavery and servitude
  [2] Warfare and conquest
  [3] Trade and commerce
[1/4] Fetching Wikipedia document...
  -> trying: fundamental principle xenia ancient greek culture
  ✓ Fetched: Hellenism (modern religion) (65371 chars)
[2/4] Cleaning Wikipedia text...
  ✓ Cleaned text (34375 chars)
[3/4] Scoring semantic matches...
  A: raw=0.301282, norm=0.416224 -> Hospitality and generosity towards strangers
  B: raw=0.135298, norm=0.186916 -> Slavery and servitude
  C: raw=0.137462, norm=0.189906 -> Warfare and conquest
  D: raw=0.149803, norm=0.206954 -> Trade and commerce
[4/4] Submitting answer 0 (A)...
  ✓ CORRECT! Earned: $100.00
  ✓ Saved to: /content/gdrive/MyDrive/NLP_assignment/semantic_game_test/Q01_A

QUESTION 2 | Level 2
Q: What term

In [130]:
# Ensemble game test: combine chunking + whole-text semantic search.
# This blends the chunk-based signal with a full cleaned-document semantic signal,
# then answers the game using the combined confidence.

import os
import json
import re
from datetime import datetime
from urllib.parse import quote_plus
from urllib.request import Request, urlopen
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

ENSEMBLE_TEST_DIR = "/content/gdrive/MyDrive/NLP_assignment/ensemble_game_test"
os.makedirs(ENSEMBLE_TEST_DIR, exist_ok=True)

STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "been", "but", "by", "for", "from",
    "has", "have", "how", "in", "is", "it", "its", "of", "on", "or", "that", "the",
    "their", "there", "these", "this", "those", "to", "was", "were", "what", "when",
    "where", "which", "who", "why", "with", "during", "did", "do", "does", "according",
    "following", "among", "between", "into", "than", "then", "through", "under", "over",
    "after", "before", "period", "status", "purpose", "reason", "used", "use"
}


def extract_keywords(text: str) -> list[str]:
    words = re.findall(r"[a-z0-9]+", text.lower())
    keywords = [word for word in words if len(word) > 2 and word not in STOPWORDS]
    seen = set()
    ordered_keywords = []
    for word in keywords:
        if word not in seen:
            ordered_keywords.append(word)
            seen.add(word)
    return ordered_keywords


def fetch_wikipedia_raw_document(query: str):
    search_url = (
        "https://en.wikipedia.org/w/api.php"
        "?action=query"
        "&list=search"
        f"&srsearch={quote_plus(query)}"
        "&srlimit=5"
        "&format=json"
        "&utf8=1"
        "&redirects=1"
    )
    with urlopen(Request(search_url, headers={"User-Agent": "Mozilla/5.0"})) as response:
        search_data = json.load(response)

    search_results = search_data.get("query", {}).get("search", [])
    if not search_results:
        raise ValueError(f"No Wikipedia page found for query: {query}")

    query_terms = set(extract_keywords(query))
    best_result = None
    best_overlap = -1
    for result in search_results:
        title_terms = set(extract_keywords(result.get("title", "")))
        overlap = len(query_terms & title_terms)
        if overlap > best_overlap:
            best_overlap = overlap
            best_result = result

    title = best_result["title"] if best_result else search_results[0]["title"]
    raw_url = (
        "https://en.wikipedia.org/w/api.php"
        "?action=query"
        "&prop=revisions"
        "&rvprop=content"
        "&rvslots=main"
        f"&titles={quote_plus(title)}"
        "&formatversion=2"
        "&format=json"
        "&utf8=1"
        "&redirects=1"
    )
    with urlopen(Request(raw_url, headers={"User-Agent": "Mozilla/5.0"})) as response:
        page_data = json.load(response)

    page = page_data["query"]["pages"][0]
    revisions = page.get("revisions", [])
    if not revisions:
        raise ValueError(f"No raw content available for Wikipedia page: {title}")

    raw_document = revisions[0]["slots"]["main"]["content"]
    return title, raw_document


def clean_wikipedia_raw_text(raw_text: str) -> str:
    text = raw_text
    text = re.sub(r"<ref[^>/]*?>.*?</ref>", " ", text, flags=re.IGNORECASE | re.DOTALL)
    text = re.sub(r"<ref[^>]*/?>", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"<!--.*?-->", " ", text, flags=re.DOTALL)
    text = re.sub(r"\{\{[^{}]*\}\}", " ", text)
    text = re.sub(r"\[\[(?:File|Image):[^\]]*\]\]", " ", text, flags=re.IGNORECASE)

    def replace_link(match):
        target = match.group(1).strip()
        label = match.group(2)
        if label:
            return label.strip()
        if ":" in target:
            return " "
        return target.split("|")[-1].strip()

    text = re.sub(r"\[\[(.+?)(?:\|(.+?))?\]\]", replace_link, text)
    text = text.replace("[[", " ").replace("]]", " ")
    text = re.sub(r"'''''(.*?)'''''", r"\1", text)
    text = re.sub(r"'''(.*?)'''", r"\1", text)
    text = re.sub(r"''(.*?)''", r"\1", text)
    text = re.sub(r"^=+\s*(.*?)\s*=+$", r"\1", text, flags=re.MULTILINE)
    text = re.sub(r"^\s*[\*#;:]+\s*", "", text, flags=re.MULTILINE)
    text = re.sub(r"\{\|.*?\|\}", " ", text, flags=re.DOTALL)
    text = text.replace("{|", " ").replace("|}", " ")
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    return text.strip()


def chunk_text(text: str, chunk_size: int = 1400, overlap: int = 250):
    cleaned_text = re.sub(r"\s+", " ", text).strip()
    if not cleaned_text:
        return []
    step = max(chunk_size - overlap, 1)
    chunks = []
    for start in range(0, len(cleaned_text), step):
        chunk = cleaned_text[start:start + chunk_size].strip()
        if chunk:
            chunks.append(chunk)
        if start + chunk_size >= len(cleaned_text):
            break
    return chunks


def option_labels():
    return ["A", "B", "C", "D"]


def semantic_scores_for_windows(question_text: str, options: list[str], windows: list[str]):
    question_keywords = [word for word in re.findall(r"[a-z0-9]+", question_text.lower()) if len(word) > 2]
    option_queries = [f"{option_text} {' '.join(question_keywords[:6])}".strip() for option_text in options]

    word_vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1)
    char_vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=1)

    word_matrix = word_vectorizer.fit_transform(option_queries + windows)
    char_matrix = char_vectorizer.fit_transform(option_queries + windows)

    option_word = word_matrix[:len(options)]
    window_word = word_matrix[len(options):]
    option_char = char_matrix[:len(options)]
    window_char = char_matrix[len(options):]

    raw_scores = {}
    details = {}

    for option_index, option_text in enumerate(options):
        word_similarities = cosine_similarity(option_word[option_index:option_index + 1], window_word).flatten()
        char_similarities = cosine_similarity(option_char[option_index:option_index + 1], window_char).flatten()
        similarities = (0.45 * word_similarities) + (0.55 * char_similarities)
        best_window_index = int(similarities.argmax())
        best_similarity = float(similarities[best_window_index])
        label = option_labels()[option_index]
        raw_scores[label] = best_similarity
        details[label] = {
            "best_window": windows[best_window_index],
            "best_similarity": best_similarity,
            "top_windows": [
                {
                    "window": windows[idx],
                    "similarity": float(similarities[idx]),
                    "word_similarity": float(word_similarities[idx]),
                    "char_similarity": float(char_similarities[idx]),
                }
                for idx in similarities.argsort()[::-1][:5]
            ],
        }

    total = sum(raw_scores.values()) or 1.0
    normalized = {label: score / total for label, score in raw_scores.items()}
    return raw_scores, normalized, details


print("=" * 80)
print("STARTING ENSEMBLE GAME TEST")
print("=" * 80)

ensemble_game = client.game.start(competition_id=comp_id)
print(f"Game started. Session ID: {ensemble_game.session_id}")
print(f"Total questions: {ensemble_game.state.competition.max_levels}")
print()

question_num = 0
total_correct = 0

while ensemble_game.in_progress:
    question_num += 1
    question = ensemble_game.current_question
    if not question:
        print("No question available. Game ended.")
        break

    question_text = question.text
    options = [opt.text for opt in question.options]

    print(f"\n{'=' * 80}")
    print(f"QUESTION {question_num} | Level {ensemble_game.current_level}")
    print(f"Q: {question_text}")
    for i, opt in enumerate(options):
        print(f"  [{i}] {opt}")

    print("[1/5] Fetching Wikipedia document...")
    candidate_queries = []
    query_words = extract_keywords(question_text)
    if query_words:
        candidate_queries.append(" ".join(query_words[:8]))
    candidate_queries.append(question_text)

    wiki_title = "N/A"
    wiki_raw_document = ""
    last_error = None
    for candidate_query in candidate_queries:
        try:
            print(f"  -> trying: {candidate_query}")
            wiki_title, wiki_raw_document = fetch_wikipedia_raw_document(candidate_query)
            if wiki_raw_document:
                break
        except Exception as e:
            last_error = e

    if not wiki_raw_document:
        print(f"  ❌ No usable Wikipedia page found. Last error: {last_error}")
        clean_wiki_text = ""
    else:
        print(f"  ✓ Fetched: {wiki_title} ({len(wiki_raw_document)} chars)")
        print("[2/5] Cleaning Wikipedia text...")
        clean_wiki_text = clean_wikipedia_raw_text(wiki_raw_document)
        print(f"  ✓ Cleaned text ({len(clean_wiki_text)} chars)")

    print("[3/5] Chunking and semantic scoring...")
    if clean_wiki_text:
        chunks = chunk_text(clean_wiki_text)
        if not chunks:
            chunks = [clean_wiki_text]
        chunk_raw, chunk_norm, chunk_details = semantic_scores_for_windows(question_text, options, chunks)

        # Also score the whole cleaned document as one window.
        whole_raw, whole_norm, whole_details = semantic_scores_for_windows(question_text, options, [clean_wiki_text])

        ensemble_raw = {}
        ensemble_norm = {}
        for label in option_labels():
            ensemble_raw[label] = (0.5 * chunk_raw[label]) + (0.5 * whole_raw[label])
        ensemble_total = sum(ensemble_raw.values()) or 1.0
        ensemble_norm = {label: score / ensemble_total for label, score in ensemble_raw.items()}

        best_option = max(ensemble_norm, key=ensemble_norm.get)
        best_option_idx = option_labels().index(best_option)
    else:
        chunks = []
        chunk_raw = {"A": 0.0, "B": 0.0, "C": 0.0, "D": 0.0}
        chunk_norm = {"A": 0.25, "B": 0.25, "C": 0.25, "D": 0.25}
        chunk_details = {}
        whole_raw = {"A": 0.0, "B": 0.0, "C": 0.0, "D": 0.0}
        whole_norm = {"A": 0.25, "B": 0.25, "C": 0.25, "D": 0.25}
        whole_details = {}
        ensemble_raw = {"A": 0.0, "B": 0.0, "C": 0.0, "D": 0.0}
        ensemble_norm = {"A": 0.25, "B": 0.25, "C": 0.25, "D": 0.25}
        best_option = "A"
        best_option_idx = 0

    print("Chunk-based scores:")
    for label, option_text in zip(option_labels(), options):
        print(f"  {label}: raw={chunk_raw[label]:.6f}, norm={chunk_norm[label]:.6f} -> {option_text}")

    print("Whole-text scores:")
    for label, option_text in zip(option_labels(), options):
        print(f"  {label}: raw={whole_raw[label]:.6f}, norm={whole_norm[label]:.6f} -> {option_text}")

    print("Ensemble scores:")
    for label, option_text in zip(option_labels(), options):
        print(f"  {label}: raw={ensemble_raw[label]:.6f}, norm={ensemble_norm[label]:.6f} -> {option_text}")

    print(f"[4/5] Submitting answer {best_option_idx} ({best_option})...")
    result = ensemble_game.answer(best_option_idx)
    if result.correct:
        total_correct += 1
        print(f"  ✓ CORRECT! Earned: ${result.earned_amount:,.2f}")
    else:
        print("  ✗ WRONG!")

    question_dir = os.path.join(ENSEMBLE_TEST_DIR, f"Q{question_num:02d}_{best_option}")
    payload = {
        "question_number": question_num,
        "level": ensemble_game.current_level,
        "question": question_text,
        "options": options,
        "wiki_title": wiki_title,
        "wiki_queries_tried": candidate_queries,
        "clean_wiki_length": len(clean_wiki_text),
        "chunk_raw": chunk_raw,
        "chunk_norm": chunk_norm,
        "whole_raw": whole_raw,
        "whole_norm": whole_norm,
        "ensemble_raw": ensemble_raw,
        "ensemble_norm": ensemble_norm,
        "best_option": best_option,
        "best_option_index": best_option_idx,
        "chunk_details": chunk_details,
        "whole_details": whole_details,
        "correct": result.correct,
        "earned": float(result.earned_amount),
        "timestamp": datetime.now().isoformat(),
    }
    os.makedirs(question_dir, exist_ok=True)
    with open(os.path.join(question_dir, "ensemble_question.json"), "w", encoding="utf-8") as handle:
        json.dump(payload, handle, ensure_ascii=False, indent=2)
    print(f"  ✓ Saved to: {question_dir}")

    if result.game_over:
        print(f"\n{'=' * 80}")
        print("GAME OVER!")
        print(f"Questions answered: {question_num}")
        print(f"Correct answers: {total_correct}")
        print(f"Accuracy: {100 * total_correct / question_num:.1f}%")
        print(f"Final earnings: ${result.earned_amount:,.2f}")
        print(f"Output directory: {ENSEMBLE_TEST_DIR}")
        print(f"{'=' * 80}")
        break

print("Ensemble game test completed!")

STARTING ENSEMBLE GAME TEST
Game started. Session ID: 8849
Total questions: 15


QUESTION 1 | Level 1
Q: What was the primary reason the ancient Egyptians developed Egyptian blue?
  [0] To imitate the precious stones turquoise and lapis lazuli
  [1] To produce a pigment for medical purposes
  [2] To decorate the exterior of their pyramids
  [3] To create a new form of glass
[1/5] Fetching Wikipedia document...
  -> trying: primary ancient egyptians developed egyptian blue
  ✓ Fetched: Egyptian blue (36136 chars)
[2/5] Cleaning Wikipedia text...
  ✓ Cleaned text (21953 chars)
[3/5] Chunking and semantic scoring...
Chunk-based scores:
  A: raw=0.188430, norm=0.389930 -> To imitate the precious stones turquoise and lapis lazuli
  B: raw=0.092460, norm=0.191333 -> To produce a pigment for medical purposes
  C: raw=0.100715, norm=0.208415 -> To decorate the exterior of their pyramids
  D: raw=0.101636, norm=0.210322 -> To create a new form of glass
Whole-text scores:
  A: raw=0.130549, norm

In [133]:
# RAG game test: retrieve relevant chunks, then generate an answer from those chunks.
# This uses retrieval to build a focused context, then asks the existing text-generation pipeline
# to choose one option index (0, 1, 2, or 3).

import os
import json
import re
from datetime import datetime
from urllib.parse import quote_plus
from urllib.request import Request, urlopen
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

RAG_TEST_DIR = "/content/gdrive/MyDrive/NLP_assignment/rag_game_test"
os.makedirs(RAG_TEST_DIR, exist_ok=True)

STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "been", "but", "by", "for", "from",
    "has", "have", "how", "in", "is", "it", "its", "of", "on", "or", "that", "the",
    "their", "there", "these", "this", "those", "to", "was", "were", "what", "when",
    "where", "which", "who", "why", "with", "during", "did", "do", "does", "according",
    "following", "among", "between", "into", "than", "then", "through", "under", "over",
    "after", "before", "period", "status", "purpose", "reason", "used", "use"
}


def extract_keywords(text: str) -> list[str]:
    words = re.findall(r"[a-z0-9]+", text.lower())
    keywords = [word for word in words if len(word) > 2 and word not in STOPWORDS]
    seen = set()
    ordered_keywords = []
    for word in keywords:
        if word not in seen:
            ordered_keywords.append(word)
            seen.add(word)
    return ordered_keywords


def fetch_wikipedia_raw_document(query: str):
    search_url = (
        "https://en.wikipedia.org/w/api.php"
        "?action=query"
        "&list=search"
        f"&srsearch={quote_plus(query)}"
        "&srlimit=5"
        "&format=json"
        "&utf8=1"
        "&redirects=1"
    )
    with urlopen(Request(search_url, headers={"User-Agent": "Mozilla/5.0"})) as response:
        search_data = json.load(response)

    search_results = search_data.get("query", {}).get("search", [])
    if not search_results:
        raise ValueError(f"No Wikipedia page found for query: {query}")

    query_terms = set(extract_keywords(query))
    best_result = None
    best_overlap = -1
    for result in search_results:
        title_terms = set(extract_keywords(result.get("title", "")))
        overlap = len(query_terms & title_terms)
        if overlap > best_overlap:
            best_overlap = overlap
            best_result = result

    title = best_result["title"] if best_result else search_results[0]["title"]
    raw_url = (
        "https://en.wikipedia.org/w/api.php"
        "?action=query"
        "&prop=revisions"
        "&rvprop=content"
        "&rvslots=main"
        f"&titles={quote_plus(title)}"
        "&formatversion=2"
        "&format=json"
        "&utf8=1"
        "&redirects=1"
    )
    with urlopen(Request(raw_url, headers={"User-Agent": "Mozilla/5.0"})) as response:
        page_data = json.load(response)

    page = page_data["query"]["pages"][0]
    revisions = page.get("revisions", [])
    if not revisions:
        raise ValueError(f"No raw content available for Wikipedia page: {title}")

    raw_document = revisions[0]["slots"]["main"]["content"]
    return title, raw_document


def clean_wikipedia_raw_text(raw_text: str) -> str:
    text = raw_text
    text = re.sub(r"<ref[^>/]*?>.*?</ref>", " ", text, flags=re.IGNORECASE | re.DOTALL)
    text = re.sub(r"<ref[^>]*/?>", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"<!--.*?-->", " ", text, flags=re.DOTALL)
    text = re.sub(r"\{\{[^{}]*\}\}", " ", text)
    text = re.sub(r"\[\[(?:File|Image):[^\]]*\]\]", " ", text, flags=re.IGNORECASE)

    def replace_link(match):
        target = match.group(1).strip()
        label = match.group(2)
        if label:
            return label.strip()
        if ":" in target:
            return " "
        return target.split("|")[-1].strip()

    text = re.sub(r"\[\[(.+?)(?:\|(.+?))?\]\]", replace_link, text)
    text = text.replace("[[", " ").replace("]]", " ")
    text = re.sub(r"'''''(.*?)'''''", r"\1", text)
    text = re.sub(r"'''(.*?)'''", r"\1", text)
    text = re.sub(r"''(.*?)''", r"\1", text)
    text = re.sub(r"^=+\s*(.*?)\s*=+$", r"\1", text, flags=re.MULTILINE)
    text = re.sub(r"^\s*[\*#;:]+\s*", "", text, flags=re.MULTILINE)
    text = re.sub(r"\{\|.*?\|\}", " ", text, flags=re.DOTALL)
    text = text.replace("{|", " ").replace("|}", " ")
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    return text.strip()


def chunk_text(text: str, chunk_size: int = 1400, overlap: int = 250):
    cleaned_text = re.sub(r"\s+", " ", text).strip()
    if not cleaned_text:
        return []
    step = max(chunk_size - overlap, 1)
    chunks = []
    for start in range(0, len(cleaned_text), step):
        chunk = cleaned_text[start:start + chunk_size].strip()
        if chunk:
            chunks.append(chunk)
        if start + chunk_size >= len(cleaned_text):
            break
    return chunks


def retrieve_top_chunks(question_text: str, clean_text: str, top_k: int = 5):
    chunks = chunk_text(clean_text)
    if not chunks:
        return [], []

    question_keywords = extract_keywords(question_text)
    query_text = question_text + " " + " ".join(question_keywords[:6])

    vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1)
    matrix = vectorizer.fit_transform([query_text] + chunks)
    query_vector = matrix[0:1]
    chunk_vectors = matrix[1:]
    similarities = cosine_similarity(query_vector, chunk_vectors).flatten()

    ranked_indices = similarities.argsort()[::-1][:top_k]
    top_chunks = [chunks[index] for index in ranked_indices]
    top_scores = [float(similarities[index]) for index in ranked_indices]
    return top_chunks, top_scores


def build_rag_prompt(question_text: str, options: list[str], retrieved_chunks: list[str]) -> str:
    options_block = "\n".join([f"{idx}: {option}" for idx, option in enumerate(options)])
    context_block = "\n\n".join([f"[CHUNK {i+1}] {chunk}" for i, chunk in enumerate(retrieved_chunks)])
    return f"""You are answering a multiple-choice question using only the retrieved context.
Return only one digit: 0, 1, 2, or 3.

Question: {question_text}

Options:
{options_block}

Retrieved context:
{context_block}
"""


def parse_answer_from_text(text: str) -> int:
    match = re.search(r"\b([0-3])\b", text)
    if match:
        return int(match.group(1))
    lowered = text.lower()
    if "option a" in lowered or "answer a" in lowered:
        return 0
    if "option b" in lowered or "answer b" in lowered:
        return 1
    if "option c" in lowered or "answer c" in lowered:
        return 2
    if "option d" in lowered or "answer d" in lowered:
        return 3
    return 0


def get_generation_pipeline():
    if "pipe" in globals():
        return pipe
    if "llm_pipeline" in globals():
        return llm_pipeline
    if "model" in globals() and "tokenizer" in globals():
        return pipeline("text-generation", model=model, tokenizer=tokenizer)
    return None


def rag_answer(question_text: str, options: list[str], clean_text: str):
    retrieved_chunks, retrieval_scores = retrieve_top_chunks(question_text, clean_text, top_k=5)
    prompt = build_rag_prompt(question_text, options, retrieved_chunks)
    generator = get_generation_pipeline()

    if generator is None:
        # Fallback to a retrieval score if no text-generation pipeline is available.
        best_idx = 0
        best_score = -1.0
        for idx, option in enumerate(options):
            option_query = f"{option} {' '.join(extract_keywords(question_text)[:6])}".strip()
            local_vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1)
            local_matrix = local_vectorizer.fit_transform([option_query] + retrieved_chunks)
            option_vector = local_matrix[0:1]
            chunk_vectors = local_matrix[1:]
            similarities = cosine_similarity(option_vector, chunk_vectors).flatten()
            score = float(similarities.max()) if len(similarities) else 0.0
            if score > best_score:
                best_score = score
                best_idx = idx
        return best_idx, prompt, retrieved_chunks, retrieval_scores, "fallback_retrieval"

    output = generator(
        prompt,
        max_new_tokens=64,
        do_sample=False,
        temperature=0.0,
    )

    generated_text = ""
    if isinstance(output, list) and output:
        first_item = output[0]
        if isinstance(first_item, dict):
            if isinstance(first_item.get("generated_text"), str):
                generated_text = first_item["generated_text"]
            elif isinstance(first_item.get("generated_text"), list) and first_item["generated_text"]:
                last_message = first_item["generated_text"][-1]
                if isinstance(last_message, dict):
                    generated_text = last_message.get("content", "")

    best_idx = parse_answer_from_text(generated_text)
    return best_idx, prompt, retrieved_chunks, retrieval_scores, generated_text or "generator_output"


print("=" * 80)
print("STARTING RAG GAME TEST")
print("=" * 80)

rag_game = client.game.start(competition_id=comp_id)
print(f"Game started. Session ID: {rag_game.session_id}")
print(f"Total questions: {rag_game.state.competition.max_levels}")
print()

question_num = 0
total_correct = 0

while rag_game.in_progress:
    question_num += 1
    question = rag_game.current_question
    if not question:
        print("No question available. Game ended.")
        break

    question_text = question.text
    options = [opt.text for opt in question.options]

    print(f"\n{'=' * 80}")
    print(f"QUESTION {question_num} | Level {rag_game.current_level}")
    print(f"Q: {question_text}")
    for i, opt in enumerate(options):
        print(f"  [{i}] {opt}")

    print("[1/4] Fetching Wikipedia document...")
    candidate_queries = []
    query_words = extract_keywords(question_text)
    if query_words:
        candidate_queries.append(" ".join(query_words[:8]))
    candidate_queries.append(question_text)

    wiki_title = "N/A"
    wiki_raw_document = ""
    last_error = None
    for candidate_query in candidate_queries:
        try:
            print(f"  -> trying: {candidate_query}")
            wiki_title, wiki_raw_document = fetch_wikipedia_raw_document(candidate_query)
            if wiki_raw_document:
                break
        except Exception as e:
            last_error = e

    if not wiki_raw_document:
        print(f"  ❌ No usable Wikipedia page found. Last error: {last_error}")
        clean_wiki_text = ""
    else:
        print(f"  ✓ Fetched: {wiki_title} ({len(wiki_raw_document)} chars)")
        print("[2/4] Cleaning Wikipedia text...")
        clean_wiki_text = clean_wikipedia_raw_text(wiki_raw_document)
        print(f"  ✓ Cleaned text ({len(clean_wiki_text)} chars)")

    print("[3/4] Retrieving context and generating answer...")
    answer_idx, rag_prompt, retrieved_chunks, retrieval_scores, rag_source = rag_answer(question_text, options, clean_wiki_text)
    print(f"  RAG source: {rag_source}")
    print(f"  Retrieved chunks: {len(retrieved_chunks)}")
    for i, score in enumerate(retrieval_scores, 1):
        print(f"    chunk {i} retrieval score: {score:.6f}")
    print("  Prompt preview:")
    print(rag_prompt[:800])
    print(f"[4/4] Submitting answer {answer_idx}...")

    result = rag_game.answer(answer_idx)
    if result.correct:
        total_correct += 1
        print(f"  ✓ CORRECT! Earned: ${result.earned_amount:,.2f}")
    else:
        print("  ✗ WRONG!")

    question_dir = os.path.join(RAG_TEST_DIR, f"Q{question_num:02d}_{answer_idx}")
    payload = {
        "question_number": question_num,
        "level": rag_game.current_level,
        "question": question_text,
        "options": options,
        "wiki_title": wiki_title,
        "wiki_queries_tried": candidate_queries,
        "clean_wiki_length": len(clean_wiki_text),
        "retrieved_chunks": retrieved_chunks,
        "retrieval_scores": retrieval_scores,
        "rag_source": rag_source,
        "rag_prompt": rag_prompt,
        "predicted_answer_index": answer_idx,
        "predicted_answer": options[answer_idx] if options else "",
        "correct": result.correct,
        "earned": float(result.earned_amount),
        "timestamp": datetime.now().isoformat(),
    }
    os.makedirs(question_dir, exist_ok=True)
    with open(os.path.join(question_dir, "rag_question.json"), "w", encoding="utf-8") as handle:
        json.dump(payload, handle, ensure_ascii=False, indent=2)
    print(f"  ✓ Saved to: {question_dir}")

    if result.game_over:
        print(f"\n{'=' * 80}")
        print("GAME OVER!")
        print(f"Questions answered: {question_num}")
        print(f"Correct answers: {total_correct}")
        print(f"Accuracy: {100 * total_correct / question_num:.1f}%")
        print(f"Final earnings: ${result.earned_amount:,.2f}")
        print(f"Output directory: {RAG_TEST_DIR}")
        print(f"{'=' * 80}")
        break

print("RAG game test completed!")

STARTING RAG GAME TEST
Game started. Session ID: 8907
Total questions: 15


QUESTION 1 | Level 1
Q: What was Nero's primary method for gaining popularity among the lower-class citizens of Rome?
  [0] Promoting public entertainments
  [1] Reducing taxes for the wealthy
  [2] Building new temples
  [3] Increasing military spending
[1/4] Fetching Wikipedia document...
  -> trying: nero primary method gaining popularity lower class citizens


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  ✓ Fetched: Gladiator (120041 chars)
[2/4] Cleaning Wikipedia text...
  ✓ Cleaned text (71658 chars)
[3/4] Retrieving context and generating answer...
  RAG source: You are answering a multiple-choice question using only the retrieved context.
Return only one digit: 0, 1, 2, or 3.

Question: What was Nero's primary method for gaining popularity among the lower-class citizens of Rome?

Options:
0: Promoting public entertainments
1: Reducing taxes for the wealthy
2: Building new temples
3: Increasing military spending

Retrieved context:
[CHUNK 1] ator types, and the Bignor Roman Villa mosaic from Provincial Britain shows Cupids as gladiators. Souvenir ceramics were produced depicting named gladiators in combat; similar images of higher quality, were available on more expensive articles in high quality ceramic, glass or silver. Some of the best preserved gladiator graffiti are from Pompeii and Herculaneum, in public areas including Pompeii's Forum and amphitheater, and in the private re

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  ✓ Fetched: Sumerian language (276591 chars)
[2/4] Cleaning Wikipedia text...
  ✓ Cleaned text (143970 chars)
[3/4] Retrieving context and generating answer...
  RAG source: You are answering a multiple-choice question using only the retrieved context.
Return only one digit: 0, 1, 2, or 3.

Question: What term describes the Akkadian representation of the original Sumerian name for Babylon?

Options:
0: Babilla
1: Kan dig̃irak
2: Bābilim
3: Ká.Dig̃ir.Ra

Retrieved context:
[CHUNK 1] Old Sumerian period, while Northern Sumerian only had /i/-. Later Southern Sumerian generalized /i/- as well. In Southern Sumerian, the conjugation prefix expressing the passive was 𒁀 ba-, while in Northern Sumerian, it was 𒀀 a-. In Southern Sumerian after the Old Akkadian period, the conjugation prefix 𒀀 a-, which had originally existed in both dialects, disappears entirely apart from the variant 𒀠 al-, which only appears in subordinate clauses. In Southern Sumerian, the Old Sumerian phoneme ř merged with 